# PyIBS Example 2: Maximum-likelihood estimation with PyBADS

In this example, we fit the orientation discrimination model of the [previous notebook](./pyibs_example_1_basic_usage.ipynb) to its data set by maximum-likelihood estimation, with the optimizer PyBADS and an IBS estimate of the negative log-likelihood as its noisy target, and we compare the result with the exact maximum-likelihood point.

This notebook is Part 2 of a series of notebooks in which we present example usages of IBS with the PyIBS package. The code used in this example is available as a script [here](https://github.com/acerbilab/pyibs/blob/main/examples/scripts/pyibs_example_2_maximum_likelihood_with_pybads.py).

In [1]:
import math

import numpy as np
from pybads import BADS
from scipy.optimize import minimize

from pyibs import IBS
from pyibs.examples.psycho_model import psycho_generator, psycho_neg_logl

## 0. Why PyBADS?

An IBS estimate is noisy: two calls at the same parameter vector give two different values. Most optimizers assume a deterministic target, and a noisy one makes them stop early or wander. Bayesian Adaptive Direct Search (BADS), whose Python implementation is PyBADS, is designed for the noisy targets of model fitting, and takes the SD of each evaluation when the target can estimate it, as IBS does. PyBADS and PyVBMC, used in the [next notebook](./pyibs_example_3_posterior_with_pyvbmc.ipynb), are among the lab's [tools for fitting models to data](https://acerbilab.org/model-fitting/).

## 1. The model, the data and the bounds

The model is the orientation discrimination model of the previous notebook, with parameter vector $\theta = (\eta, \mu, \gamma)$: the log of the sensory noise's SD, the bias and the lapse rate. We draw the data set as MATLAB IBS's `ibs_example.m` draws its own: 600 trials, whose orientations are drawn from a normal distribution of SD 3°, and the responses simulated at $\theta_\text{true} = (\log 1, 0.2, 0.03)$. Every random draw of the notebook comes from a seeded NumPy generator, so that the notebook gives the same results each time it runs. The seeds are spawned from one root seed, which gives each random stream its own seed, independent of the others; the data set is the same in every notebook of this series.

In [2]:
# Independent seeds for the data, the IBS estimates, the start and PyBADS
seed_data, seed_ibs, seed_start, seed_bads = np.random.SeedSequence(
    2026
).spawn(4)

theta_true = np.array([math.log(1.0), 0.2, 0.03])  # log(sigma), bias, lapse
n_trials = 600

rng = np.random.default_rng(seed_data)
S = 3 * rng.standard_normal((n_trials, 1))  # Orientation of each trial
R = psycho_generator(theta_true, S, rng)  # Response of each trial

We take the bounds of `ibs_example.m`:

- The hard bounds `LB` and `UB` are the bounds of the optimization: PyBADS never evaluates the target outside them. The lower bound of the lapse rate, 0.01, keeps every response possible, with probability at least 0.005 on every trial. This matters for IBS, whose cost on a trial is about $1/p_i$ samples per repeat: at a lapse rate of 0, a response far from the bias would be all but impossible, and its sampling would take very long.
- The plausible bounds `PLB` and `PUB` mark the region where we expect the solution. PyBADS draws its initial points in that box, and scales its search by it.

In [3]:
LB = np.array([math.log(0.1), -2.0, 0.01])  # Lower bounds
UB = np.array([math.log(10.0), 2.0, 1.0])  # Upper bounds
PLB = np.array([math.log(0.2), -1.0, 0.02])  # Plausible lower bounds
PUB = np.array([math.log(5.0), 1.0, 0.2])  # Plausible upper bounds

Since this model has a closed-form likelihood, we can find its exact maximum-likelihood point for comparison, with a gradient-based optimizer on the exact negative log-likelihood, a smooth and deterministic target.

In [4]:
exact_fit = minimize(
    psycho_neg_logl,
    (PLB + PUB) / 2,
    args=(S, R),
    method="L-BFGS-B",
    bounds=list(zip(LB, UB)),
)
theta_ml, neg_logl_min = exact_fit.x, exact_fit.fun
print(f"Exact maximum-likelihood point: {theta_ml.round(4)}")
print(f"Exact minimum of the negative log-likelihood: {neg_logl_min:.2f}")

Exact maximum-likelihood point: [0.0584 0.2828 0.0184]
Exact minimum of the negative log-likelihood: 151.78


## 2. The IBS target

The target that PyBADS minimizes returns the IBS estimate of the *negative* log-likelihood, and its SD:

- With `options={"specify_target_noise": True}`, PyBADS takes from the target a tuple `(value, sd)`, which `additional_output="std"` returns. The second element must be the SD, not the variance (`additional_output="var"`): PyBADS reads it as the SD of the value's noise, in the value's units, and a variance would misstate the noise wherever it is far from 1 (a variance of 9 is an SD of 3).
- PyBADS works best with an SD of order 1 or less near the solution (see the [PyBADS FAQ](https://acerbilab.github.io/pybads/faq.html#faq-can-pybads-handle-any-arbitrary-amount-of-noise-in-the-objective)), which for this data set takes about 100 repeats (Section 5 of the previous notebook). Far from the solution, the SD can be larger.
- PyBADS refuses an SD of 0, which IBS reports only when every trial matched its response at its first sample, in every repeat; a data set of this size practically never does, and a call that reports it warns, with a link to the [PyIBS FAQ](https://acerbilab.github.io/pyibs/faq.html#faq-why-is-the-sd-of-the-estimate-zero-and-why-do-pybads-and-pyvbmc-refuse-it).

In [5]:
ibs = IBS(psycho_generator, R, S, vectorized=True, random_seed=seed_ibs)


def target(theta):
    """IBS estimate of the negative log-likelihood, and its SD."""
    return ibs(theta, num_reps=100, additional_output="std")

As `ibs_example.m` does, we start the optimization from a point drawn at random in the plausible box. Here is the target there, and the exact value:

In [6]:
x0 = PLB + np.random.default_rng(seed_start).random(3) * (PUB - PLB)
value, sd = target(x0)
print(f"Starting point: {x0.round(4)}")
print(f"Target at the start: {value:.2f} +/- {sd:.2f}")
print(f"Exact value:         {psycho_neg_logl(x0, S, R):.2f}")

Starting point: [ 0.1637 -0.5321  0.1351]
Target at the start: 192.54 +/- 1.24
Exact value:         194.46


## 3. Running PyBADS

We create the `BADS` object with `specify_target_noise` and a `random_seed`, which makes PyBADS's own random draws reproducible; the target's draws come from the generator of the `IBS` object, which its own seed fixes. `optimize()` runs the optimization and prints its progress, and may print a short tip with a link to PyBADS's documentation (`options["show_tips"] = False` turns the tips off).

In [7]:
bads = BADS(
    target,
    x0,
    LB,
    UB,
    PLB,
    PUB,
    options={"specify_target_noise": True, "random_seed": seed_bads},
)
optimize_result = bads.optimize()

Variables (index) internally transformed to log coordinates: [2].
Beginning optimization of a STOCHASTIC objective function (specified noise)

Tip: If you work with a coding agent, give it the PyBADS skill from GitHub, or copy the skills/pybads folder of PyBADS's repository into the agent's skill directory: it points the agent to the parts of PyBADS's documentation relevant to your task.
https://github.com/acerbilab/pybads/blob/main/skills/pybads/SKILL.md

 Iteration    f-count      E[f(x)]        SD[f(x)]           MeshScale          Method              Actions
     0           1         194.397         1.24537               1                                  
     0          33         156.692         1.24537               1          Initial mesh            Initial points
     1          38         156.692         1.10304             0.5          Refine grid             Train
     2          39         153.892         1.04331             0.5      Successful search (ES-wcm)        
  

## 4. Results

`optimize_result["x"]` is the solution, and `optimize_result["fval"]` PyBADS's estimate of the target there: the precision-weighted mean of its final evaluations at the solution, with its standard error `optimize_result["fsd"]`.

In [8]:
theta_bads = optimize_result["x"]
fval, fsd = optimize_result["fval"], optimize_result["fsd"]
print(f"PyBADS estimate at its solution: {fval:.2f} +/- {fsd:.2f}")
print(f"Target evaluations: {optimize_result['func_count']}")
print()
print(f"{'':17}{'eta':>8}{'bias':>8}{'lapse':>8}")
for name, theta in [
    ("Generating", theta_true),
    ("Exact ML point", theta_ml),
    ("PyBADS solution", theta_bads),
]:
    print(f"{name:17}" + "".join(f"{x:8.4f}" for x in theta))

PyBADS estimate at its solution: 151.87 +/- 0.33
Target evaluations: 343

                      eta    bias   lapse
Generating         0.0000  0.2000  0.0300
Exact ML point     0.0584  0.2828  0.0184
PyBADS solution    0.0448  0.2634  0.0242


Once the optimization is over, it is good practice, with any noisy target, to estimate the target at the solution with a higher precision than during the search: here with 1,000 repeats, ten times as many as the target's, as `ibs_example.m` takes ten times as many repeats as its own target. The closed form gives the exact values.

In [9]:
neg_logl, neg_logl_sd = ibs(theta_bads, num_reps=1000, additional_output="std")
neg_logl_exact = psycho_neg_logl(theta_bads, S, R)
print(
    "IBS estimate at the solution (1,000 repeats): "
    f"{neg_logl:.2f} +/- {neg_logl_sd:.2f}"
)
print(f"Exact value at the solution:  {neg_logl_exact:.2f}")
print(f"Exact minimum:                {neg_logl_min:.2f}")
print(f"Difference:                   {neg_logl_exact - neg_logl_min:.2f}")

IBS estimate at the solution (1,000 repeats): 152.43 +/- 0.33
Exact value at the solution:  151.89
Exact minimum:                151.78
Difference:                   0.11


### Remarks

- The exact negative log-likelihood at PyBADS's solution exceeds the exact minimum by much less than 1, the SD of the target's evaluations: PyBADS found the maximum-likelihood point to within the precision of its target.
- The maximum-likelihood point differs from the generating parameters because the data set is finite; it would approach them as the number of trials grows.
- In practice, run PyBADS from several starting points, and keep the best solution, as the [PyBADS FAQ](https://acerbilab.github.io/pybads/faq.html#faq-how-do-i-run-pybads-from-several-starting-points) explains; compare their values at the higher precision.

## 5. Conclusions

In this notebook, we have fitted a model by maximum-likelihood estimation with PyBADS, from IBS estimates of its negative log-likelihood and their SD, without using its likelihood, and found its exact maximum-likelihood point to within the precision of the estimates. The [next notebook](./pyibs_example_3_posterior_with_pyvbmc.ipynb) infers the posterior over its parameters, and its model evidence, with PyVBMC.

## References

1. van Opheusden, B., Acerbi, L. & Ma, W. J. (2020). Unbiased and efficient log-likelihood estimation with inverse binomial sampling. *PLOS Computational Biology* 16(12): e1008483. [https://doi.org/10.1371/journal.pcbi.1008483](https://doi.org/10.1371/journal.pcbi.1008483)
2. Singh, G. S. & Acerbi, L. (2024). PyBADS: Fast and robust black-box optimization in Python. *Journal of Open Source Software* 9(94): 5694. [https://doi.org/10.21105/joss.05694](https://doi.org/10.21105/joss.05694)
3. Acerbi, L. & Ma, W. J. (2017). Practical Bayesian optimization for model fitting with Bayesian adaptive direct search. In *Advances in Neural Information Processing Systems 30*: 1834-1844. [https://arxiv.org/abs/1705.04405](https://arxiv.org/abs/1705.04405)